Name - Aniket Rajendra Deshpande

ASU ID - 1233954685

Course - CSE 572 Data Mining Fall 2025 C - Homework 3

Task 1 - Algorithmic Analysis K-Means Clustering with Real World Dataset 


Q1 - Run K-means clustering with Euclidean, Cosine and Jarcard similarity. Specify K= the number of categorical values of y (the number of classifications). Compare the SSEs of Euclidean-K-means, Cosine-K-means, Jarcard-K-means. Which method is better? (10 points)


In [ ]:
import numpy as np

# Loading the data

X = np.loadtxt("data.csv", delimiter=",")
y = np.loadtxt("label.csv", delimiter=",", dtype=int)

n_samples, n_features = X.shape
K = len(np.unique(y))   
# number of clusters = number of unique labels
print(f"Data shape: {X.shape}, #classes (K) = {K}")

# Distance functions
def euclidean_distance_matrix(X, C):
    """
    Compute pairwise Euclidean distances between all points in X and centroids C.
    X: (n_samples, d)
    C: (k, d)
    Returns:
        dist:  (n_samples, k)  distances
        dist2: (n_samples, k)  squared distances
    """
    X_sq = np.sum(X ** 2, axis=1, keepdims=True)      
    # (n,1)
    C_sq = np.sum(C ** 2, axis=1)                     
    # (k,)
    XC = X @ C.T                                      
    # (n,k)
    d2 = X_sq + C_sq - 2 * XC                         
    # ||x||^2 + ||c||^2 - 2 x·c
    d2 = np.clip(d2, 0, None)                        
     # numerical safety
    return np.sqrt(d2), d2


def cosine_distance_matrix(X, C, eps=1e-10):
    """
    1 - cosine similarity.
    Cosine sim = (x·c)/(||x|| ||c||) -> distance = 1 - cos.
    """
    X_norm = np.linalg.norm(X, axis=1, keepdims=True) + eps   
    # (n,1)
    C_norm = np.linalg.norm(C, axis=1) + eps                  
    # (k,)
    dot = X @ C.T                                             
    # (n,k)
    cos = dot / (X_norm * C_norm)
    cos = np.clip(cos, -1.0, 1.0)                             
    # numerical safety
    dist = 1.0 - cos
    return dist, dist ** 2


def jaccard_distance_matrix(X, C, eps=1e-10):
    """
    1 - Generalized Jaccard similarity for nonnegative vectors:
        sim(x, c) = sum_i min(x_i, c_i) / sum_i max(x_i, c_i)
    Distance = 1 - sim.
    """
    n, d = X.shape
    k = C.shape[0]
    sims = np.empty((n, k), dtype=float)

    for j in range(k):
        c = C[j]
        mn = np.minimum(X, c)      
        # (n,d)
        mx = np.maximum(X, c)      
        # (n,d)
        num = np.sum(mn, axis=1)
        den = np.sum(mx, axis=1) + eps
        sims[:, j] = num / den

    dist = 1.0 - sims
    return dist, dist ** 2

#K-means implementation

def kmeans(X, k, distance_type="euclidean", max_iters=20, random_state=0):
    """
    Basic K-means from scratch with pluggable distance function.

    distance_type: 'euclidean', 'cosine', or 'jaccard'
    Returns:
        labels:   cluster assignment for each point (n_samples,)
        centroids: (k, d)
        sse:      sum of squared distances to assigned centroids
    """
    rng = np.random.RandomState(random_state)
    n, d = X.shape

    # initialization: pick K random points as centroids
    indices = rng.choice(n, size=k, replace=False)
    centroids = X[indices].copy()

    for it in range(max_iters):
        # computing distances
        if distance_type == "euclidean":
            dist, dist2 = euclidean_distance_matrix(X, centroids)
        elif distance_type == "cosine":
            dist, dist2 = cosine_distance_matrix(X, centroids)
        elif distance_type == "jaccard":
            dist, dist2 = jaccard_distance_matrix(X, centroids)
        else:
            raise ValueError("Unknown distance_type")

        # assigning points to closest centroid
        labels = np.argmin(dist, axis=1)

        # recomputing centroids as mean of assigned points
        new_centroids = np.zeros_like(centroids)
        for j in range(k):
            mask = (labels == j)
            if np.any(mask):
                new_centroids[j] = X[mask].mean(axis=0)
            else:
                # if a cluster is empty, reinitialize its centroid randomly
                new_centroids[j] = X[rng.randint(0, n)]

        # checking convergence
        if np.allclose(new_centroids, centroids):
            # centroids no longer changing significantly
            centroids = new_centroids
            print(f"[{distance_type}] converged at iteration {it}")
            break

        centroids = new_centroids

    # final SSE computation
    if distance_type == "euclidean":
        dist, dist2 = euclidean_distance_matrix(X, centroids)
    elif distance_type == "cosine":
        dist, dist2 = cosine_distance_matrix(X, centroids)
    else:
        dist, dist2 = jaccard_distance_matrix(X, centroids)

    labels = np.argmin(dist, axis=1)
    sse = np.sum(dist2[np.arange(n), labels])

    return labels, centroids, sse

#Running K-means for all three distances


print("\nRunning Euclidean K-means...")
labels_e, C_e, sse_e = kmeans(X, K, distance_type="euclidean", max_iters=20, random_state=42)
print(f"Euclidean K-means SSE = {sse_e:.4f}")

print("\nRunning Cosine K-means...")
labels_c, C_c, sse_c = kmeans(X, K, distance_type="cosine", max_iters=20, random_state=42)
print(f"Cosine K-means SSE = {sse_c:.4f}")

print("\nRunning Jaccard K-means...")
labels_j, C_j, sse_j = kmeans(X, K, distance_type="jaccard", max_iters=20, random_state=42)
print(f"Jaccard K-means SSE = {sse_j:.4f}")

#comparison of SSE values

print("\n SSE Comparison")
print(f"Euclidean SSE : {sse_e:.4f}")
print(f"Cosine SSE    : {sse_c:.4f}")
print(f"Jaccard SSE   : {sse_j:.4f}")

# which method is better based purely on SSE:
sse_dict = {
    "Euclidean": sse_e,
    "Cosine": sse_c,
    "Jaccard": sse_j
}
best_method = min(sse_dict, key=sse_dict.get)
print(f"\nMethod with smallest SSE (according to this metric): {best_method}")


Data shape: (10000, 784), #classes (K) = 10

Running Euclidean K-means...
Euclidean K-means SSE = 25419980937.7006

Running Cosine K-means...
Cosine K-means SSE = 688.1586

Running Jaccard K-means...
Jaccard K-means SSE = 3664.7975

=== SSE Comparison ===
Euclidean SSE : 25419980937.7006
Cosine SSE    : 688.1586
Jaccard SSE   : 3664.7975

Method with smallest SSE (according to this metric): Cosine


Q2 - Compare the accuracies of Euclidean-K-means Cosine-K-means, Jarcard-K-means. First, label each cluster using the majority vote label of the data points in that cluster. Later, compute the predictive accuracy of Euclidean-K-means, Cosine-K-means, Jarcard-K-means. Which metric is better? (10 points)

In [ ]:
from collections import Counter
import numpy as np

def compute_accuracy(true_labels, cluster_labels, K):
    """
    Assigning each cluster a label via majority vote,
    then compute the overall clustering accuracy.
    """
    cluster_to_label = {}

    for c in range(K):
        # Find true labels of points belonging to cluster c
        indices = np.where(cluster_labels == c)[0]
        if len(indices) == 0:
            cluster_to_label[c] = -1  # empty cluster fallback
            continue
        true_vals = true_labels[indices]

        # Majority vote
        most_common_label = Counter(true_vals).most_common(1)[0][0]
        cluster_to_label[c] = most_common_label

    # Predict labels for all samples
    predicted = np.array([cluster_to_label[c] for c in cluster_labels])

    # Accuracy = % of correct predictions
    accuracy = np.mean(predicted == true_labels)
    return accuracy


#Computing accuracy for each distance metric

acc_euclidean = compute_accuracy(y, labels_e, K)
acc_cosine = compute_accuracy(y, labels_c, K)
acc_jaccard = compute_accuracy(y, labels_j, K)

print("\nAccuracy Comparison")
print(f"Euclidean-K-means Accuracy : {acc_euclidean:.4f}")
print(f"Cosine-K-means Accuracy    : {acc_cosine:.4f}")
print(f"Jaccard-K-means Accuracy   : {acc_jaccard:.4f}")

# Which is best?
acc_dict = {
    "Euclidean": acc_euclidean,
    "Cosine": acc_cosine,
    "Jaccard": acc_jaccard
}
best_metric = max(acc_dict, key=acc_dict.get)
print(f"\nBest accuracy metric: {best_metric}")



 Accuracy Comparison
Euclidean-K-means Accuracy : 0.5858
Cosine-K-means Accuracy    : 0.6120
Jaccard-K-means Accuracy   : 0.5763

Best accuracy metric: Cosine


Q3 - Set up the same stop criteria: “when there is no change in centroid position OR when the SSE value increases in the next iteration OR when the maximum preset value (e.g., 500, you can set the preset value by yourself) of iteration is complete”, for Euclidean-K-means, Cosine-K means, Jarcard-K-means. Which method requires more iterations and times to converge? (10 points)

In [3]:
def kmeans_Q3(X, k, distance_type="euclidean", max_iters=500, random_state=0):
    """
    K-means with Q3 stopping criteria:
    - Stop when centroid positions do not change
    - OR when SSE increases
    - OR when iteration reaches max_iters
    Returns:
        labels
        centroids
        final_sse
        iteration_count
    """

    rng = np.random.RandomState(random_state)
    n, d = X.shape

    # Initialize centroids
    indices = rng.choice(n, size=k, replace=False)
    centroids = X[indices].copy()

    prev_sse = None
    iteration = 0

    for iteration in range(1, max_iters + 1):

        # Compute distances
        if distance_type == "euclidean":
            dist, dist2 = euclidean_distance_matrix(X, centroids)
        elif distance_type == "cosine":
            dist, dist2 = cosine_distance_matrix(X, centroids)
        elif distance_type == "jaccard":
            dist, dist2 = jaccard_distance_matrix(X, centroids)
        else:
            raise ValueError("Unknown distance_type")

        # Assign clusters
        labels = np.argmin(dist, axis=1)

        # Compute SSE
        sse = np.sum(dist2[np.arange(n), labels])

        # Stop if SSE increases
        if prev_sse is not None and sse > prev_sse:
            # print(f"[{distance_type}] SSE increased; stopping at iter {iteration}")
            break

        prev_sse = sse

        #Recomputing centroids
        new_centroids = np.zeros_like(centroids)
        for j in range(k):
            pts = X[labels == j]
            if len(pts) > 0:
                new_centroids[j] = pts.mean(axis=0)
            else:
                new_centroids[j] = X[rng.randint(0, n)]

        # Stopping if centroids do not change
        if np.allclose(new_centroids, centroids):
            # print(f"[{distance_type}] Centroids stable; stopping at iter {iteration}")
            centroids = new_centroids
            break

        centroids = new_centroids

    # Final SSE
    if distance_type == "euclidean":
        _, dist2 = euclidean_distance_matrix(X, centroids)
    elif distance_type == "cosine":
        _, dist2 = cosine_distance_matrix(X, centroids)
    else:
        _, dist2 = jaccard_distance_matrix(X, centroids)

    labels = np.argmin(_, axis=1)
    final_sse = np.sum(dist2[np.arange(n), labels])

    return labels, centroids, final_sse, iteration


In [4]:
print("\nRunning Euclidean K-means (Q3 criteria)...")
labels_e2, C_e2, sse_e2, it_e = kmeans_Q3(X, K, "euclidean")
print(f"Euclidean: iterations = {it_e}, SSE = {sse_e2:.4f}")

print("\nRunning Cosine K-means (Q3 criteria)...")
labels_c2, C_c2, sse_c2, it_c = kmeans_Q3(X, K, "cosine")
print(f"Cosine: iterations = {it_c}, SSE = {sse_c2:.4f}")

print("\nRunning Jaccard K-means (Q3 criteria)...")
labels_j2, C_j2, sse_j2, it_j = kmeans_Q3(X, K, "jaccard")
print(f"Jaccard: iterations = {it_j}, SSE = {sse_j2:.4f}")

print("\n Iteration Comparison")
print(f"Euclidean iterations: {it_e}")
print(f"Cosine iterations   : {it_c}")
print(f"Jaccard iterations  : {it_j}")



Running Euclidean K-means (Q3 criteria)...
Euclidean: iterations = 48, SSE = 25321064456.8186

Running Cosine K-means (Q3 criteria)...
Cosine: iterations = 46, SSE = 684.0879

Running Jaccard K-means (Q3 criteria)...
Jaccard: iterations = 12, SSE = 3686.0758

 Iteration Comparison
Euclidean iterations: 48
Cosine iterations   : 46
Jaccard iterations  : 12


Q4 - Compare the SSEs of Euclidean-K-means Cosine-K-means, Jarcard-K-means with respect to the following three terminating conditions: (10 points) 
• when there is no change in centroid position 
• when the SSE value increases in the next iteration 
• when the maximum preset value (e.g., 100) of iteration is complete 


In [ ]:
import numpy as np

def kmeans_with_stop(X, k, distance_type="euclidean", stop_mode="centroid", max_iters=100, random_state=0):

    rng = np.random.RandomState(random_state)
    n, d = X.shape

    # Initialize centroids
    indices = rng.choice(n, size=k, replace=False)
    centroids = X[indices].copy()

    prev_sse = None
    iterations = 0

    for it in range(1, max_iters + 1):
        iterations = it

        # Compute distances
        if distance_type == "euclidean":
            dist, dist2 = euclidean_distance_matrix(X, centroids)
        elif distance_type == "cosine":
            dist, dist2 = cosine_distance_matrix(X, centroids)
        elif distance_type == "jaccard":
            dist, dist2 = jaccard_distance_matrix(X, centroids)
        else:
            raise ValueError("Unknown distance_type")

        labels = np.argmin(dist, axis=1)
        sse = np.sum(dist2[np.arange(n), labels])

        # Stop if SSE increases
        if stop_mode == "sse_increase":
            if prev_sse is not None and sse > prev_sse:
                break

        prev_sse = sse

        # Recompute centroids
        new_centroids = np.zeros_like(centroids)
        for j in range(k):
            pts = X[labels == j]
            if len(pts) > 0:
                new_centroids[j] = pts.mean(axis=0)
            else:
                new_centroids[j] = X[rng.randint(0, n)]

        # top if centroids don't change
        if stop_mode == "centroid":
            if np.allclose(new_centroids, centroids):
                centroids = new_centroids
                break

        centroids = new_centroids

    # Final SSE with final centroids
    if distance_type == "euclidean":
        dist, dist2 = euclidean_distance_matrix(X, centroids)
    elif distance_type == "cosine":
        dist, dist2 = cosine_distance_matrix(X, centroids)
    else:
        dist, dist2 = jaccard_distance_matrix(X, centroids)

    labels = np.argmin(dist, axis=1)
    final_sse = np.sum(dist2[np.arange(n), labels])

    return labels, centroids, final_sse, iterations


In [ ]:
stop_modes = ["centroid", "sse_increase", "max_iter"]
stop_names = {"centroid": "No change in centroids", "sse_increase": "SSE increases", "max_iter": "Max iteration = 100"}

distance_types = ["euclidean", "cosine", "jaccard"]

results = {}

for mode in stop_modes:
    print(f"\n Stop condition: {stop_names[mode]}")
    results[mode] = {}
    for dist_type in distance_types:
        labels, C, sse, iters = kmeans_with_stop(
            X, K,
            distance_type=dist_type,
            stop_mode=mode,
            max_iters=100,
            random_state=42
        )
        results[mode][dist_type] = (sse, iters)
        print(f"{dist_type.capitalize():9s} -> iterations = {iters:3d}, SSE = {sse:.4f}")



=== Stop condition: No change in centroids ===
Euclidean -> iterations =  33, SSE = 25414767689.9612
Cosine    -> iterations =  48, SSE = 686.4356
Jaccard   -> iterations =  59, SSE = 3660.3895

=== Stop condition: SSE increases ===
Euclidean -> iterations = 100, SSE = 25414767689.9612
Cosine    -> iterations =  29, SSE = 686.2293
Jaccard   -> iterations =   2, SSE = 4239.9465

=== Stop condition: Max iteration = 100 ===
Euclidean -> iterations = 100, SSE = 25414767689.9612
Cosine    -> iterations = 100, SSE = 686.4356
Jaccard   -> iterations = 100, SSE = 3660.3895
